In [8]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager

options = Options()
# Argumentos de estabilidad para Docker
options.add_argument("--headless")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--disable-gpu")

try:
    # El Manager instala el driver compatible con la versión de Chrome
    driver = webdriver.Chrome(
        service=Service(ChromeDriverManager().install()),
        options=options
    )

    driver.get("https://www.google.com")
    print(f"CONECTADO! Título de la página: {driver.title}")
    driver.quit()

except Exception as e:
    print(f"❌ Error: {e}")

CONECTADO! Título de la página: Google


In [25]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

# ==========================================================
# 1. URL
# ==========================================================
URL_OBJETIVO = "https://www.yapo.cl/region_metropolitana/autos"

headers = {
    "User-Agent": "Mozilla/5.0"
}

# ==========================================================
# 2. CONEXIÓN
# ==========================================================
print("🔄 Conectando...")
response = requests.get(URL_OBJETIVO, headers=headers)

soup = BeautifulSoup(response.text, "html.parser")

dataset_final = []

# ==========================================================
# 3. SCRAPING REAL
# ==========================================================
items = soup.select("div.d3-ad-tile")

print(f"📊 Encontrados: {len(items)}")

for item in items:
    try:
        # NOMBRE
        nombre = item.select_one("a.d3-ad-tile__description").text.strip()

        # PRECIO (puede no existir en todos)
        precio_tag = item.select_one("span.price")
        precio = precio_tag.text.strip() if precio_tag else "0"

        precio_limpio = "".join(filter(str.isdigit, precio))

        dataset_final.append({
            "variable_nombre": nombre,
            "variable_valor": precio_limpio,
            "status": 1.0
        })

    except:
        continue

# ==========================================================
# 4. SALIDA
# ==========================================================
if dataset_final:
    df = pd.DataFrame(dataset_final)
    df.to_csv("datos_extraidos_grupo.csv", index=False)

    print("✅ Proceso exitoso")
    print(df.head())
else:
    print("⚠️ No se capturaron datos")
    

🔄 Conectando...
📊 Encontrados: 0
⚠️ No se capturaron datos
